# 1. Import Libraries

In [1]:
import pandas as pd 
import re
from pathlib import Path

# 2. Merge Metadata Files

In [2]:
METADATA_FOLDER = Path.cwd() / 'metadata_list'

metadata_df_list = []

for path_to_file in METADATA_FOLDER.glob('*.csv'):

    print(path_to_file)
    
    sub_metadata_df = pd.read_csv(path_to_file)

    metadata_df_list.append(sub_metadata_df)

metadata_df = pd.concat(metadata_df_list, ignore_index=True)

C:\Users\young\Jupyter Notebook\metadata_list\fcc_news_release_metadata_page_0-149.csv
C:\Users\young\Jupyter Notebook\metadata_list\fcc_news_release_metadata_page_150-249.csv
C:\Users\young\Jupyter Notebook\metadata_list\fcc_news_release_metadata_page_250-331.csv


In [3]:
for col in ['released_on', 'issued_on', 'adopted']:
    metadata_df[col] = pd.to_datetime(metadata_df[col], errors='coerce')

metadata_df = metadata_df.sort_values(
    'released_on', 
    ascending=False, 
    ignore_index=True
)

In [4]:
metadata_df['download_status'].value_counts()

download_status
downloaded                 3200
skipped_no_matching_txt     111
Name: count, dtype: int64

In [5]:
metadata_df = metadata_df[metadata_df['download_status'] == 'downloaded']

In [6]:
metadata_df['downloaded_filename'] = metadata_df['downloaded_files'].apply(lambda x: Path(str(x)).name)

metadata_df['downloaded_filename_key'] = metadata_df['downloaded_filename'].str.lower()

mask_duplicates = metadata_df['downloaded_filename_key'].duplicated(keep='first')

metadata_df = metadata_df[~mask_duplicates]

# 3. Extract the Text Body from News Releases

## 3.1. Regular Expressions for News Release Formats

In [7]:
DATELINE_LINE_RE = re.compile(
    r'''(?ix)
    ^\(?\s*
    WASHINGTON
    (?:\s*,?\s*(?:D\.?\s*C\.?))?
    [),]?\s*
    (?:,?\s*[A-Z]{3,9}\.?\s+\d{1,2},\s+\d{4})?
    \s*[:—–\-�]\s*
    '''
)

In [8]:
DATE_LINE_RE = re.compile(
    r'''(?ix)
    ^
    (?:FOR\s+IMMEDIATE\s+RELEASE:?\s*)?
    (?:[A-Z]{3,9}\.?\s+\d{1,2},\s+\d{4})\b
    '''
)

In [9]:
HEADER_RE = re.compile(
    r'''(?ix)
    ^(?:
        NEWS|
        ADVISORY|
        PUBLIC\s+NOTICE|
        Federal\s+Communications\s+Commission|
        445\s+12(?:th)?\s+Street|
        th$|
        Street,\s*S\.W\.|
        Washington,?\s*D\.?\s*C\.?\s*20554|
        This\s+is\s+an\s+unofficial\s+announcement|
        constitutes\s+official\s+action|
        See\s+MCI\s+v\.\s*FCC|
        News\s+Media\s+Information|
        Internet:|
        TTY:|
        Fax-On-Demand|
        ftp\.fcc\.gov|
        FOR\s+IMMEDIATE\s+RELEASE|
        NEWS\s+(?:MEDIA\s+)?CONTACTS?:?|
        MEDIA\s+CONTACT:?
    )
    '''
)

In [10]:
CONTACT_RE = re.compile(
    r'''(?ix)
    (
        \bmedia\s+contact\b|
        \bnews\s+contact\b|
        ^contact\b|
        \bemail\b|
        \be-mail\b|
        @fcc\.gov|
        \(?202\)?[- .]?418|
        888[- .]?835
    )
    '''
)

In [11]:
FOOTER_RE = re.compile(
    r'''(?ix)
    ^\s*(?:\#\s*){3}\s*$|
    ^\s*[-–—]*\s*FCC\s*[-–—]*\s*$
    '''
)

In [12]:
TITLE_SKIP_RE = re.compile(
    r'''(?ix)
    ^(?:
        [-–—]+|
        \(?[A-Z]{1,4}\s+DOCKET\s+NO\.?|
        Docket\s+No\.?|
        Re\s*:
    )
    '''
)

## 3.2. Define Helper Functions

In [13]:
def normalize_lines(text):
    '''
    Convert raw text into clean non-empty lines.
    '''
    text = (
        text.replace('\ufeff', '')
            .replace('\r\n', '\n')
            .replace('\r', '\n')
            .replace('\xa0', ' ')
    )

    lines = []

    for line in text.splitlines():
        line = re.sub(r'\s+', ' ', line).strip()

        if line:
            lines.append(line)

    return lines

In [14]:
def upper_ratio(line):
    '''
    Calculate how much of a line is uppercase.
    Useful for detecting title/header lines.
    '''
    letters = [char for char in line if char.isalpha()]

    if not letters:
        return 0

    return sum(char.isupper() for char in letters) / len(letters)

In [15]:
def is_front_matter_line(line):
    '''
    Detect header/title/contact lines before the actual body.
    '''
    if HEADER_RE.search(line):
        return True

    if CONTACT_RE.search(line):
        return True

    if DATE_LINE_RE.search(line):
        return True

    if TITLE_SKIP_RE.search(line):
        return True

    # Many FCC titles are written in uppercase.
    if len(line) <= 180 and upper_ratio(line) >= 0.72 and not line.endswith(('.', '?', '!')):
        return True

    return False

In [16]:
def looks_like_body_start(line):
    '''
    Detect a likely first body line.
    '''
    if is_front_matter_line(line):
        return False

    if len(line) < 20:
        return False

    # A normal sentence-like line.
    if re.search(r'[a-z].*[\.;?!]$', line):
        return True

    # Some PDF-extracted lines do not end with punctuation because of line wrapping.
    if re.match(
        r'(?i)(The|Today|This|In|On|At|As|Chairman|Commissioner|Consumers|Broadband|Wireless|Cable|FCC|Without)\b',
        line
    ):
        return True

    return False

In [17]:
def find_body_end(lines):
    '''
    Find the end of the substantive body before footer/contact blocks.
    '''
    for i, line in enumerate(lines):
        if FOOTER_RE.match(line):
            return i, 'footer_marker'

    # Some recent releases use trailing media contact blocks.
    for i, line in enumerate(lines):
        if i > len(lines) * 0.65 and re.match(r'(?i)Media Contact:', line):
            return i, 'trailing_media_contact'

        if i > len(lines) * 0.65 and re.match(r'(?i)Office of Media Relations:', line):
            return i, 'office_media_footer'

    return len(lines), 'no_footer_marker'

In [18]:
def clean_join(lines):
    '''
    Join wrapped PDF/text lines into one clean text string.
    '''
    body = ' '.join(lines)

    # Remove spaces before punctuation.
    body = re.sub(r'\s+([,.;:?!])', r'\1', body)

    # Collapse repeated spaces.
    body = re.sub(r'\s+', ' ', body).strip()

    return body

In [19]:
def extract_body(text):
    '''
    Extract the main body of one FCC news release.

    Parameters
    ----------
    text : str
        Raw text from one .txt file.

    Returns
    -------
    body_text : str
    extraction_method : str
    footer_method : str
    '''
    lines = normalize_lines(text)

    end_idx, footer_method = find_body_end(lines)
    lines = lines[:end_idx]

    # Method 1: Most news releases have a WASHINGTON dateline.
    for i, line in enumerate(lines):
        if DATELINE_LINE_RE.match(line):
            body_lines = lines[i:]

            body_lines[0] = DATELINE_LINE_RE.sub('', body_lines[0]).strip()
            body_lines = [line for line in body_lines if line]

            return clean_join(body_lines), 'dateline', footer_method

    # Method 2: If no dateline exists, skip front matter/title/contact lines.
    i = 0

    while i < min(60, len(lines)) and is_front_matter_line(lines[i]):
        i += 1

    for j in range(i, len(lines)):
        if looks_like_body_start(lines[j]):
            return clean_join(lines[j:]), 'fallback_body_like_after_front_matter', footer_method

    # Method 3: Last fallback.
    for j, line in enumerate(lines):
        if looks_like_body_start(line):
            return clean_join(lines[j:]), 'fallback_body_like_global', footer_method

    return clean_join(lines[i:]), 'fallback_after_front_matter', footer_method

In [20]:
def build_body_dataframe_from_folder(text_folder):
    text_folder = Path(text_folder)

    records = []

    for file_path in sorted(text_folder.glob('*.txt')):
        try:
            raw_text = file_path.read_text(encoding='utf-8-sig')
        except UnicodeDecodeError:
            raw_text = file_path.read_text(encoding='cp1252', errors='replace')

        body_text, extraction_method, footer_method = extract_body(raw_text)

        records.append({
            'filename': file_path.name,
            'body_text': body_text,
            'body_word_count': len(body_text.split()),
            'extraction_method': extraction_method,
            'footer_method': footer_method,
            'needs_review': len(body_text.split()) < 30
        })

    return pd.DataFrame(records)

## 3.3. Collect the Text

In [21]:
TEXT_FOLDER = Path('fcc_news_release_documents')

body_df = build_body_dataframe_from_folder(TEXT_FOLDER)

body_df.head()

,filename,body_text,body_word_count,extraction_method,footer_method,needs_review
0,100105_Federal_and_State_Partners_To_Test_Nati...,The Department of Homeland Security�s Federal ...,559,dateline,footer_marker,False
1,100105_MEDIA_BUREAU_ANNOUNCES_PANELISTS_AND_AG...,The Media Bureau today announced the panelists...,270,dateline,footer_marker,False
2,100106_PANELISTS_ANNOUNCED_FOR_JANUARY_13_WORK...,The Federal Communications Commission will hol...,288,dateline,no_footer_marker,False
3,100107_FCC_Launches_Reboot.FCC.gov_to_Engage_P...,"- Today, the Federal Communications Commission...",334,dateline,footer_marker,False
4,"100107_Statement_of_William_T._Lake,_Chief,_Me...",Today Sinclair and Mediacom have completed the...,82,dateline,footer_marker,False


In [22]:
print('Number of documents:', len(body_df))
print('Empty body texts:', (body_df['body_word_count'] == 0).sum())
print('Documents needing review:', body_df['needs_review'].sum())

print('\n', body_df['extraction_method'].value_counts())

print('\n', body_df['footer_method'].value_counts())

print('\n', body_df['needs_review'].value_counts())   

Number of documents: 3196
Empty body texts: 1
Documents needing review: 12

 extraction_method
dateline                                 2924
fallback_body_like_after_front_matter     251
fallback_after_front_matter                21
Name: count, dtype: int64

 footer_method
footer_marker          3032
no_footer_marker        163
office_media_footer       1
Name: count, dtype: int64

 needs_review
False    3184
True       12
Name: count, dtype: int64


In [23]:
body_df.to_csv(
    'fcc_news_release_body_texts.csv',
    index=False,
    encoding='utf-8-sig'
)

In [24]:
body_df['filename_key'] = body_df['filename'].str.lower()

metadata_df = metadata_df.merge(
    body_df,
    left_on='downloaded_filename_key',
    right_on='filename_key',
    how='left'
)

metadata_df = metadata_df.drop(columns=[
    'selected_txt_count', 'download_status', 'error_message',
    'downloaded_files', 'downloaded_filename', 'downloaded_filename_key', 'filename_key'
    ]
)

In [25]:
metadata_df

,page_title,full_title,document_type,bureaus,description,webpage_url,selected_txt_urls,all_txt_urls,all_txt_count,released_on,issued_on,adopted,filename,body_text,body_word_count,extraction_method,footer_method,needs_review
0,FCC Proposes to Amend Audible Crawl Rule to Pr...,FCC Proposes to Amend Audible Crawl Rule to Pr...,News Release,Media Relations; Space,NaN,https://www.fcc.gov/document/fcc-proposes-amen...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,3,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Proposes_to_Amend_Audible_Crawl_Rul...,"Today, the Federal Communications Commission a...",292,dateline,footer_marker,False
1,FCC Adopts Rules to Enhance the Integrity of t...,FCC Adopts Rules to Enhance the Integrity of t...,News Release,Media Relations; Wireline Competition,NaN,https://www.fcc.gov/document/fcc-adopts-rules-...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,4,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Adopts_Rules_to_Enhance_the_Integri...,"Today, the Federal Communications Commission a...",342,dateline,footer_marker,False
2,FCC Targets 'Covered List' Entities' Blanket S...,FCC Proposes Banning Entities on 'Covered List...,News Release,Media Relations; Wireline Competition,NaN,https://www.fcc.gov/document/fcc-targets-cover...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,3,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Targets_Covered_List_Entities_Blank...,"Today, the Federal Communications Commission v...",346,dateline,footer_marker,False
3,FCC Targets Device Test Labs in Nations Withou...,FCC Looks to Prohibit Electronic Device Testin...,News Release,Engineering & Technology; Media Relations,NaN,https://www.fcc.gov/document/fcc-targets-devic...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,3,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Targets_Device_Test_Labs_in_Nations...,"Today, the Federal Communications Commission v...",410,dateline,footer_marker,False
4,FCC Proposes Strengthening 'Know-Your-Customer...,FCC Proposes Strengthening 'Know-Your-Customer...,News Release,Consumer and Governmental Affairs; Media Relat...,NaN,https://www.fcc.gov/document/fcc-proposes-stre...,https://docs.fcc.gov/public/attachments/DOC-42...,https://docs.fcc.gov/public/attachments/DOC-42...,3,2026-04-30,2026-04-30,2026-04-30,260430_FCC_Proposes_Strengthening_Know-Your-Cu...,In an effort to stop illegal calls before they...,303,dateline,footer_marker,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3191,"Statement of William T. Lake, Chief, Media Bur...","Statement of William T. Lake, Chief, Media Bur...",News Release,Media,NaN,https://www.fcc.gov/document/statement-william...,https://docs.fcc.gov/public/attachments/DOC-29...,https://docs.fcc.gov/public/attachments/DOC-29...,1,2010-01-07,2010-01-07,NaT,"100107_Statement_of_William_T._Lake,_Chief,_Me...",Today Sinclair and Mediacom have completed the...,82,dateline,footer_marker,False
3192,FCC Launches Reboot.FCC.gov to Engage Public i...,FCC Launches Reboot.FCC.gov to Engage Public i...,News Release,Office of the Chairman; Media Relations,NaN,https://www.fcc.gov/document/fcc-launches-rebo...,https://docs.fcc.gov/public/attachments/DOC-29...,https://docs.fcc.gov/public/attachments/DOC-29...,1,2010-01-07,2010-01-07,NaT,100107_FCC_Launches_Reboot.FCC.gov_to_Engage_P...,"- Today, the Federal Communications Commission...",334,dateline,footer_marker,False
3193,PANELISTS ANNOUNCED FOR JANUARY 13 WORKSHOP ON...,PANELISTS ANNOUNCED FOR JANUARY 13 WORKSHOP ON...,News Release,Wireline Competition,NaN,https://www.fcc.gov/document/panelists-announc...,https://docs.fcc.gov/public/attachments/DOC-29...,https://docs.fcc.gov/public/attachments/DOC-29...,1,2010-01-06,2010-01-06,NaT,100106_PANELISTS_ANNOUNCED_FOR_JANUARY_13_WORK...,The Federal Communications